Testing Agents

In [1]:
from IPython.display import Markdown
from pprint import pprint

from langchain_aws import ChatBedrockConverse
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [2]:
llm_config = {
    'model': "us.anthropic.claude-sonnet-4-6",
    'region_name': "us-east-1"
}

In [3]:
system_prompt = '''
You are an expert log analyzer. 
You will be given a set of logs and your task is to analyze them and provide a summary of the issues found.

Along with the summary, you have provide the explanation of the issues found in the logs.
After providing the issues you suggest the possible solutions to fix the issues found in the logs.
'''

test_logs = '''
INFO  - 2024/07/09-15:51:00,963 UTC - OITTOR56APCC140 - TD0374503.86242.01.c07441b.02218.Mux.Svr.4438  - TcServer tcserver00033@CADENAS_QA1_TcPoolA@3392@OITTOR56APCC140 removed.
INFO  - 2024/07/09-15:51:00,993 UTC - OITTOR56APCC140 - TD0374503.86242.01.c07441b.02218.Mux.Svr.4438  - TcServer tcserver00033@CADENAS_QA1_TcPoolA@3392@OITTOR56APCC140 finished.
WARN  - 2024/07/09-20:36:19,378 UTC - OITTOR56APCC140 - CADENAS_QA1_TcPoolA.38.DataStoreChangePoller   - Failed to query the database: SELECT TN.* FROM CLUSTERCONFIGURATIONS_602633619_863071946 TN. Closed Connection. The query will be retried.
ERROR - 2024/07/09-20:36:19,385 UTC - OITTOR56APCC140 - CADENAS_QA1_TcPoolA.38.DataStoreChangePoller   - Failed to connect to the database jdbc:oracle:thin:@CPLMT01Q1 as user TcClusterDB. Listener refused the connection with the following error:
ORA-12514, TNS:listener does not currently know of service requested in connect descriptor
 :com.teamcenter.jeti.util.Log.error(Unknown Source) com.teamcenter.jeti.store.SqlConnection 
ERROR - 2024/07/09-20:36:22,391 UTC - OITTOR56APCC140 - CADENAS_QA1_TcPoolA.38.DataStoreChangePoller   - Failed to connect to the database jdbc:oracle:thin:@CPLMT01Q1 as user TcClusterDB. Listener refused the connection with the following error:
ORA-12514, TNS:listener does not currently know of service requested in connect descriptor
 :com.teamcenter.jeti.util.Log.error(Unknown Source) com.teamcenter.jeti.store.SqlConnection 
ERROR - 2024/07/09-20:36:25,398 UTC - OITTOR56APCC140 - CADENAS_QA1_TcPoolA.38.DataStoreChangePoller   - Failed to connect to the database jdbc:oracle:thin:@CPLMT01Q1 as user TcClusterDB. Listener refused the connection with the following error:
ORA-12514, TNS:listener does not currently know of service requested in connect descriptor
 :com.teamcenter.jeti.util.Log.error(Unknown Source) com.teamcenter.jeti.store.SqlConnection 
'''

In [4]:
template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Give me details about the issues found in these logs: {logs}"),
])

llm = ChatBedrockConverse(**llm_config)
parser = StrOutputParser()

chain = template | llm | parser

In [5]:
response = chain.invoke({'logs': test_logs})
Markdown(response)

# Log Analysis Report

## Summary of Issues

The logs reveal **two distinct issues** occurring on server `OITTOR56APCC140`:

1. A **TcServer instance removal/termination** event
2. A **critical database connectivity failure** with repeated connection errors

---

## Issue 1: TcServer Instance Removal

### Details
| Field | Value |
|-------|-------|
| **Severity** | INFO |
| **Time** | 2024/07/09 - 15:51:00 UTC |
| **Server** | `tcserver00033@CADENAS_QA1_TcPoolA` |
| **Port** | 3392 |

### Explanation
- The TcServer instance `tcserver00033` was **removed and finished** from the pool `CADENAS_QA1_TcPoolA`
- This could indicate a **planned shutdown**, a **crash**, or a **pool rebalancing event**
- While logged as INFO, this event is noteworthy as it **reduces pool capacity**

---

## Issue 2: Database Connectivity Failure ⚠️ CRITICAL

### Details
| Field | Value |
|-------|-------|
| **Severity** | ERROR |
| **Start Time** | 2024/07/09 - 20:36:19 UTC |
| **Database Host** | `CPLMT01Q1` |
| **Database User** | `TcClusterDB` |
| **Connection Type** | `jdbc:oracle:thin` |
| **Error Code** | `ORA-12514` |
| **Affected Component** | `DataStoreChangePoller` |

### Error Timeline
```
20:36:19 - WARN  → Initial query failure (Closed Connection)
20:36:19 - ERROR → First connection attempt failed (ORA-12514)
20:36:22 - ERROR → Second connection attempt failed (ORA-12514) [+3 sec retry]
20:36:25 - ERROR → Third connection attempt failed (ORA-12514) [+3 sec retry]
```

### Explanation
- **ORA-12514** error means the Oracle TNS Listener is **running but cannot find the requested service name**
- The system attempted **3 retries every ~3 seconds**, all failing
- The `DataStoreChangePoller` component is **unable to detect cluster configuration changes**, which can cause **stale or inconsistent cluster state**
- This could affect **all 38 active threads** in `CADENAS_QA1_TcPoolA`

---

## Root Cause Analysis

```
Possible Sequence of Events:
┌─────────────────────────────────────────┐
│  15:51 - TcServer instance removed      │
│          (possible maintenance/crash)    │
│                  ↓                      │
│  ~20:36 - Database service becomes      │
│           unavailable on CPLMT01Q1      │
│                  ↓                      │
│  DataStoreChangePoller fails to         │
│  query cluster configurations           │
│                  ↓                      │
│  Repeated connection retries fail       │
│  with ORA-12514                         │
└─────────────────────────────────────────┘
```

---

## Possible Solutions

### For Issue 1 (TcServer Removal)
- ✅ Verify if the server removal was **planned/intentional**
- ✅ Check if `tcserver00033` was **restarted** or needs to be restarted
- ✅ Monitor pool capacity to ensure adequate servers are available

### For Issue 2 (ORA-12514 Database Error)

#### Immediate Actions
```sql
-- Verify the Oracle service name on CPLMT01Q1
lsnrctl status

-- Check registered services
lsnrctl services
```

1. **Verify Oracle Listener Status**
   - Check if the Oracle listener is running on `CPLMT01Q1`
   - Confirm the database service name matches what's in the connection descriptor

2. **Check TNS Configuration**
   - Verify the `tnsnames.ora` configuration for `CPLMT01Q1`
   - Ensure the **service name** matches the registered Oracle service

3. **Restart Oracle Listener** (if needed)
   ```bash
   lsnrctl stop
   lsnrctl start
   ```

4. **Verify Database Instance**
   - Confirm the Oracle database instance is **started and mounted**
   - Check if the service was **unregistered** from the listener

5. **Review Connection Pool Settings**
   - Increase retry attempts and intervals in `DataStoreChangePoller`
   - Implement **circuit breaker pattern** to handle persistent failures gracefully

6. **Long-term Monitoring**
   - Set up **database health monitoring alerts** for `CPLMT01Q1`
   - Configure **automatic listener restart** policies
   - Review logs for patterns of recurring connectivity issues

---

## Severity Assessment

| Issue | Severity | Impact |
|-------|----------|--------|
| TcServer Removal | 🟡 Medium | Reduced pool capacity |
| ORA-12514 DB Error | 🔴 Critical | Cluster config polling failure, potential data inconsistency |

> **Immediate Action Required**: The database connectivity issue on `CPLMT01Q1` needs urgent attention as it directly impacts cluster configuration management and stability.

In [30]:
response

"# Log Analysis Report\n\n## Summary of Issues\n\nTwo distinct issues were identified in the logs from server **OITTOR56APCC140** on **July 9, 2024**:\n\n1. **TcServer Instance Removal** *(Low Severity)*\n2. **Oracle Database Connection Failure** *(High Severity)*\n\n---\n\n## Issue 1: TcServer Instance Removal\n\n### 📋 Details\n| Field | Value |\n|-------|-------|\n| **Severity** | INFO |\n| **Time** | 15:51:00 UTC |\n| **Server** | OITTOR56APCC140 |\n| **Component** | Mux.Svr.4438 |\n| **Affected Server** | tcserver00033@CADENAS_QA1_TcPoolA |\n\n### 🔍 Explanation\n- The TcServer instance **tcserver00033** was **removed** and subsequently **finished** within the pool `CADENAS_QA1_TcPoolA`\n- This could indicate a **planned shutdown**, a **crash**, or a **pool rebalancing** event\n- While logged as INFO, this removal could have **reduced the pool's capacity** to handle requests\n\n---\n\n## Issue 2: Oracle Database Connection Failure *(Critical)*\n\n### 📋 Details\n| Field | Value |\n|-